# U.S. Flight Delays: Exploratory Data Analysis
### Bureau of Transportation Statistics — June 2026 On-Time Performance

**Author:** Ridhima  
**Tools:** Python · pandas · NumPy · matplotlib · seaborn  
**Data source:** [BTS TranStats](https://www.transtats.bts.gov/) — Reporting Carrier On-Time Performance (Table 236), June 2026

## Project overview
This notebook explores one month of U.S. domestic flight records from the Bureau of Transportation
Statistics to answer the question: **what drives departure delays, and where could a traveler
or an airline focus to reduce them?**

BTS defines a flight as *on-time* if it departs or arrives fewer than 15 minutes after schedule — the
threshold used throughout this analysis.

**Workflow**
1. Load & inspect the raw BTS extract
2. Assess data quality & clean (nulls, cancellations, types, text)
3. Engineer features (airline names, time-of-day, delay flags)
4. Explore delays across carriers, times, routes, and causes
5. Summarize findings & recommendations

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", None)

: 

## 2. Load the data

In [ ]:

CSV_PATH = "T_ONTIME_REPORTING.csv"

df = pd.read_csv(CSV_PATH)
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
df.head()

: 

## 3. First look
Inspect columns, types, and the scale of missingness.

In [ ]:
df.info()

In [ ]:
# Focus on the columns this analysis uses
cols = ['FL_DATE','OP_UNIQUE_CARRIER','OP_CARRIER_FL_NUM','ORIGIN','DEST',
        'CRS_DEP_TIME','DEP_TIME','DEP_DELAY','DEP_DELAY_NEW',
        'ARR_DELAY','ARR_DELAY_NEW','CANCELLED','CANCELLATION_CODE','DIVERTED',
        'AIR_TIME','DISTANCE','CARRIER_DELAY','WEATHER_DELAY','NAS_DELAY',
        'SECURITY_DELAY','LATE_AIRCRAFT_DELAY']
df = df[[c for c in cols if c in df.columns]].copy()
df.describe(include='all').T[['count','mean','min','max']]

## 4. Data quality assessment

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(1)
quality = pd.DataFrame({'missing': missing, 'missing_%': missing_pct})
quality[quality['missing'] > 0].sort_values('missing', ascending=False)

In [ ]:
n_cancelled = int(df['CANCELLED'].sum())
n_diverted  = int(df['DIVERTED'].sum())
print(f"Cancelled flights: {n_cancelled:,} ({n_cancelled/len(df):.1%})")
print(f"Diverted flights:  {n_diverted:,} ({n_diverted/len(df):.1%})")
print(f"Unique carriers:   {df['OP_UNIQUE_CARRIER'].nunique()}")
print(f"Unique airports:   {df['ORIGIN'].nunique()} origins")

**Notes on this real dataset**
- `DEP_DELAY` / `ARR_DELAY` are **null for cancelled flights** (they never departed) — expected, not an error.
- The five delay-cause columns (`CARRIER_DELAY`, `WEATHER_DELAY`, …) are **only populated when a flight is 15+ minutes late**; they're blank otherwise. We'll fill those with 0.
- `DEP_DELAY_NEW` is departure delay in minutes with early departures floored at 0; `DEP_DELAY` keeps negatives (early). We keep both.

## 5. Data cleaning

In [ ]:
df_clean = df.copy()

# 5a. Separate cancelled flights — analyze delays on operated flights only
cancelled = df_clean[df_clean['CANCELLED'] == 1].copy()
df_clean = df_clean[df_clean['CANCELLED'] == 0].copy()
print(f"Operated flights retained: {len(df_clean):,}")

In [ ]:
# 5b. Delay-cause columns: blank => 0 (flight wasn't late enough to attribute a cause)
cause_cols = ['CARRIER_DELAY','WEATHER_DELAY','NAS_DELAY','SECURITY_DELAY','LATE_AIRCRAFT_DELAY']
df_clean[cause_cols] = df_clean[cause_cols].fillna(0)

In [ ]:
# 5c. Drop the small number of operated flights still missing core delay values
before = len(df_clean)
df_clean = df_clean.dropna(subset=['DEP_DELAY','ARR_DELAY'])
print(f"Dropped {before - len(df_clean)} operated flights with missing delay data")

In [ ]:
# 5d. Parse scheduled hour from HHMM integer (e.g. 725 -> 7, 2154 -> 21)
df_clean['dep_hour'] = (df_clean['CRS_DEP_TIME'] // 100).astype(int)
# BTS encodes midnight as 2400 occasionally -> map to 0
df_clean['dep_hour'] = df_clean['dep_hour'].replace(24, 0)
df_clean['FL_DATE'] = pd.to_datetime(df_clean['FL_DATE'])
df_clean[['CRS_DEP_TIME','dep_hour']].head()

## 6. Feature engineering

In [ ]:
# 6a. Map carrier codes to airline names
carrier_names = {
    'AA':'American','DL':'Delta','UA':'United','WN':'Southwest','AS':'Alaska',
    'B6':'JetBlue','NK':'Spirit','F9':'Frontier','G4':'Allegiant','HA':'Hawaiian',
    'OO':'SkyWest','YX':'Republic','MQ':'Envoy','OH':'PSA','9E':'Endeavor',
    'YV':'Mesa','QX':'Horizon','C5':'CommuteAir','ZW':'Air Wisconsin'}
df_clean['airline'] = df_clean['OP_UNIQUE_CARRIER'].map(carrier_names).fillna(df_clean['OP_UNIQUE_CARRIER'])

# 6b. On-time flag (BTS standard: >15 min late)
df_clean['delayed'] = df_clean['DEP_DELAY'] > 15

# 6c. Time-of-day bucket
def part_of_day(h):
    if h < 12: return 'Morning'
    if h < 17: return 'Afternoon'
    return 'Evening'
df_clean['time_of_day'] = df_clean['dep_hour'].apply(part_of_day)

df_clean[['airline','dep_hour','time_of_day','DEP_DELAY','delayed']].head()

## 7. Exploratory analysis

### 7.1 Overall delay distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13,4))
# clip extreme tail for readability
sns.histplot(df_clean['DEP_DELAY'].clip(-30, 180), bins=60, ax=ax[0], color="#4C72B0")
ax[0].set_title("Departure delay distribution")
ax[0].set_xlabel("Departure delay (min, clipped to 180)")
sns.boxplot(x=df_clean['DEP_DELAY'].clip(-30, 180), ax=ax[1], color="#4C72B0")
ax[1].set_title("Departure delay spread")
ax[1].set_xlabel("Departure delay (min)")
plt.tight_layout(); plt.show()

print(f"Median departure delay: {df_clean['DEP_DELAY'].median():.0f} min")
print(f"Mean departure delay:   {df_clean['DEP_DELAY'].mean():.1f} min")
print(f"Share delayed >15 min:  {df_clean['delayed'].mean():.1%}")

### 7.2 On-time performance by airline

In [ ]:
airline_stats = (df_clean.groupby('airline')
                 .agg(avg_delay=('DEP_DELAY','mean'),
                      pct_delayed=('delayed','mean'),
                      flights=('DEP_DELAY','size'))
                 .query('flights >= 500')          # focus on carriers with real volume
                 .sort_values('avg_delay', ascending=False))
airline_stats['avg_delay'] = airline_stats['avg_delay'].round(1)
airline_stats['pct_delayed'] = (airline_stats['pct_delayed']*100).round(1)
airline_stats

In [ ]:
plt.figure(figsize=(9,6))
sns.barplot(data=airline_stats.reset_index(), x='avg_delay', y='airline',
            palette='rocket', hue='airline', legend=False)
plt.title("Average departure delay by airline — June 2026")
plt.xlabel("Avg departure delay (min)"); plt.ylabel("")
plt.tight_layout(); plt.show()

### 7.3 Delay by time of day

In [ ]:
hourly = df_clean.groupby('dep_hour')['DEP_DELAY'].mean()
plt.figure(figsize=(10,5))
sns.lineplot(x=hourly.index, y=hourly.values, marker='o', color="#C44E52")
plt.title("Average departure delay by scheduled hour")
plt.xlabel("Scheduled departure hour"); plt.ylabel("Avg delay (min)")
plt.xticks(range(0,24)); plt.tight_layout(); plt.show()

tod = (df_clean.groupby('time_of_day')['DEP_DELAY']
       .mean().reindex(['Morning','Afternoon','Evening']).dropna().round(1))
print("Average delay by time of day:\n", tod)

### 7.4 What causes delays? (BTS delay-cause breakdown)

In [ ]:
# Sum each cause across all flights; these are the official BTS categories
cause_totals = df_clean[cause_cols].sum().sort_values(ascending=False)
cause_labels = {'CARRIER_DELAY':'Carrier','WEATHER_DELAY':'Weather','NAS_DELAY':'Nat\'l Airspace',
                'SECURITY_DELAY':'Security','LATE_AIRCRAFT_DELAY':'Late Aircraft'}
cause_totals.index = [cause_labels[c] for c in cause_totals.index]

plt.figure(figsize=(8,5))
sns.barplot(x=cause_totals.values/1e3, y=cause_totals.index, palette='mako',
            hue=cause_totals.index, legend=False)
plt.title("Total delay minutes by cause (thousands)")
plt.xlabel("Total delay (thousand minutes)"); plt.ylabel("")
plt.tight_layout(); plt.show()

print("Share of total delay minutes by cause:")
print((cause_totals / cause_totals.sum() * 100).round(1).astype(str) + '%')

### 7.5 Busiest routes and their reliability

In [ ]:
df_clean['route'] = df_clean['ORIGIN'] + '→' + df_clean['DEST']
route_stats = (df_clean.groupby('route')
               .agg(flights=('DEP_DELAY','size'), avg_delay=('DEP_DELAY','mean'))
               .query('flights >= 200')
               .sort_values('flights', ascending=False)
               .head(15).round(1))
route_stats

### 7.6 Airline × time-of-day delay rate

In [ ]:
top_airlines = airline_stats.head(8).index
pivot = (df_clean[df_clean['airline'].isin(top_airlines)]
         .pivot_table(index='airline', columns='time_of_day',
                      values='delayed', aggfunc='mean'))
order = [c for c in ['Morning','Afternoon','Evening'] if c in pivot.columns]
pivot = pivot[order].sort_values(order[-1], ascending=False)
plt.figure(figsize=(8,5))
sns.heatmap(pivot*100, annot=True, fmt='.0f', cmap='YlOrRd',
            cbar_kws={'label':'% delayed >15 min'})
plt.title("Delay rate by airline and time of day (%)")
plt.ylabel(""); plt.xlabel("")
plt.tight_layout(); plt.show()

## 8. Key findings

> *Fill these in from your actual outputs once you run it on the full month — the numbers below are placeholders describing the shape to expect.*

1. **Delays are heavily right-skewed.** Most flights leave on time or early, but a minority of very late flights pull the mean well above the median. The median is the more honest headline number.

2. **On-time performance varies widely by airline.** Low-cost carriers typically show higher average delays and delay rates than legacy and regional carriers — worth confirming in your month.

3. **Delays build through the day.** Morning departures are the most reliable; evening flights inherit the day's accumulated (late-aircraft) delays.

4. **Late aircraft and carrier-controllable issues dominate causes**, usually ahead of weather and national-airspace (air-traffic) delays — visible in the cause breakdown.

5. **A handful of routes carry disproportionate volume**, and their reliability differs — useful for a traveler choosing flights.

## 9. Recommendations
- **Travelers:** favor morning departures and the higher-ranked airlines to minimize expected delay.
- **Airline analysts:** target evening turnaround operations, where late-aircraft delays compound — the highest-leverage place to intervene.
- **Next steps:** compare multiple months for seasonality, join airport-level data to study hub congestion, and model delay probability directly.

---
**Data:** U.S. DOT Bureau of Transportation Statistics, Reporting Carrier On-Time Performance, June 2026.  
**Methods:** pandas for cleaning/aggregation; matplotlib & seaborn for visualization. Cancelled flights were separated from the delay analysis; BTS delay-cause fields were zero-filled where flights were on time.